# Verifiable Quantum Circuit Compilation

## v4 · DFTHC-SOS Shape Screen

A diagnostic notebook, not a submission. Public work has pushed the chemistry task (`dfthcsos`, FeMoCo) to
λ_eff ≈ 47.04, T = 16,233 Toffolis, Q = 428 with Lera's `(R, B, C) = (9, 143, 40)` factorization, and fixed-word refits
have stopped paying: ours and several public ones found nothing. The leaderboard top is ≈ 28 against the public ≈ 23.95,
a ×1.6 gap in the product of task ratios. The hypothesis of this notebook: **the only task with room for that kind of
gain is chemistry, through a different factorization shape, not through weights.** Before spending hours training a
new shape we measure, cheaply and exactly, which shapes can fit the 20 mHa two-body limit at all.

What it computes, without building a single circuit:

| Measurement | Question it answers |
|---|---|
| Spectral floor of the two-body target | Below which total copy count `R·C` is the 20 mHa limit impossible for *any* factorization? |
| Optimal error with the current angle words | With today's angle circuits unchanged, how far can `C` drop, or can one whole group `R` be removed? |
| Official resource probe | Exact T and Q the verifier charges for `C = 40 → 30` with the current circuits |
| Rough cost model | What a feasible shape would be worth on the leaderboard |

**Kaggle setup:** Accelerator *None* (CPU only), Internet off. Inputs: the competition data and
[heliosli/qcc-circuits-live](https://www.kaggle.com/datasets/heliosli/qcc-circuits-live)
(or [yoyobar/qcc-local-sota-20261031](https://www.kaggle.com/datasets/yoyobar/qcc-local-sota-20261031)). Runtime ≈ 15–30 minutes.

The factorization, circuits and the ideas this screen builds on belong to the authors in **Sources and credits**.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────────────────────
SALT = 'flexonafft-shape-screen-v1'
ERROR_LIMIT = 0.020                 # Ha, official limit for error_1 and error_2
ROUNDING_NOISE = 0.01412            # Ha, quadrature noise of 8-bit word rounding in Lera's C = 40 run (13.39 -> 19.46 mHa)
C_GRID = [40, 39, 38, 36, 34, 32, 30, 28, 25]       # copies per group, current words
DROP_C_GRID = [40, 45, 50]          # copies per remaining group after one group is removed
BCD_SWEEPS = 25                     # block-coordinate sweeps of the fixed-dictionary fit
PROBE_C = [40, 39, 36, 32, 30]      # shapes sent through the official verifier for exact T and Q
RELEASES = {'heliosli/qcc-circuits-live': 'QCC_RELEASE_MANIFEST.json',
            'yoyobar/qcc-local-sota-20261031': 'portfolio-manifest.json'}
REFERENCE_COST = 3_355_599_588.3155365          # dfthcsos sample-submission cost (ratio 1.0)
PORTFOLIO = {'invsqrt': 20.65035, 'dqi': 64.76241}  # our other two task ratios (public portfolio v1)

In [ ]:
# ── Session: paths, kickmix and the dfthcsos verifier ─────────────────────────────────────────
import os, sys, glob, json, lzma, time, math, shutil, hashlib, inspect
from pathlib import Path

NB_T0 = time.time()
ROOT = Path('/kaggle/working'); ROOT.mkdir(parents=True, exist_ok=True); os.chdir(ROOT)
INPUT = Path('/kaggle/input')
PUBLIC = ROOT / 'public_dfthcsos'
NOTES = ROOT / 'notes'; NOTES.mkdir(exist_ok=True)

def deep_find(name: str) -> list[Path]:
    return sorted(Path(p) for p in glob.glob(str(INPUT / '**' / name), recursive=True))

def inventory() -> str:
    lines = sorted(glob.glob(str(INPUT / '*'))) + sorted(glob.glob(str(INPUT / '*' / '*'))) + sorted(glob.glob(str(INPUT / '*' / '*' / '*')))
    return 'Attached inputs:\n' + ('\n'.join('  ' + p for p in lines[:60]) or '  (nothing)')

def find_competition() -> Path:
    for p in (INPUT / 'competitions' / 'google-verifiable-quantum-circuit-compilation',
              INPUT / 'google-verifiable-quantum-circuit-compilation'):
        if (p / 'verifiers').is_dir():
            return p
    hits = deep_find('verify_submission_invsqrt.py')
    if hits:
        return hits[0].parent.parent.parent
    raise FileNotFoundError('Competition data not found.\n' + inventory())

def find_release() -> tuple[str, Path]:
    for name, marker in RELEASES.items():
        hits = deep_find(marker)
        if hits:
            return name, hits[0].parent
    raise FileNotFoundError('No release dataset found. Add one of ' + ', '.join(RELEASES) + '.\n' + inventory())

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            h.update(chunk)
    return h.hexdigest()

COMP = find_competition()
RELEASE_DATASET, RELEASE = find_release()
sys.path[:0] = [str(COMP / 'kickmix'), str(COMP / 'verifiers')]
import numpy as np
import pandas as pd
import kickmix as km
from dfthcsos import verify as v_dfthcsos
from dfthcsos.scoreinfra import dfthc_math as official_math, reference
from dfthcsos.scoreinfra.circuits import register_layout
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
print('competition data :', COMP)
print('release dataset  :', RELEASE_DATASET, '->', RELEASE)
print('CPU cores        :', os.cpu_count())

## Public chemistry component

Only the `dfthcsos/` part of the release is unpacked and matched to the release manifest: fifteen angle loaders and
`submission.npz`. The official verifier then runs once on it, unmodified, to record the baseline and the verifier's own
resource breakdown.

In [ ]:
if PUBLIC.exists():
    shutil.rmtree(PUBLIC)
PUBLIC.mkdir()
manifest = json.loads((RELEASE / RELEASES[RELEASE_DATASET]).read_text())
if RELEASE_DATASET.startswith('heliosli/'):
    for item in manifest['files']:
        if not item['file'].startswith('dfthcsos/'):
            continue
        src, dst = RELEASE / item['stored_as'], PUBLIC / Path(item['file']).name
        if item['stored_as'].endswith('.xz'):
            with lzma.open(src) as fi, open(dst, 'wb') as fo:
                shutil.copyfileobj(fi, fo, 1 << 22)
        else:
            shutil.copyfile(src, dst)
        assert sha256(dst) == item['sha256'], item['file']
else:
    for p in (RELEASE / 'submission' / 'dfthcsos').iterdir():
        shutil.copyfile(p, PUBLIC / p.name)
FILES = sorted(p.name for p in PUBLIC.iterdir())
print(len(FILES), 'files:', FILES[:3], '...')
print('submission.npz sha256', sha256(PUBLIC / 'submission.npz'))

t0 = time.time()
BASE_RESULT = v_dfthcsos.verify(str(PUBLIC), salt=SALT)
print(f"official verify: {BASE_RESULT['status']} in {time.time() - t0:.0f} s")
def short(r):
    keep = {k: r[k] for k in ('status', 'score', 'toffoli_count', 'num_qubits', 'lambda_eff') if k in r}
    keep.update({k: r.get('metrics', {}).get(k) for k in ('error_1', 'error_2', 'Lambda', 'E_gap')})
    return keep
print(json.dumps(short(BASE_RESULT), indent=1, default=float))
print('resources:', json.dumps(BASE_RESULT.get('resources'), indent=1, default=str)[:4000])
(NOTES / 'baseline_verify.json').write_text(json.dumps(BASE_RESULT, indent=1, default=str))

## The scorer's cost model

The prices of state preparation and rotations are fixed functions in the verifier's `scoreinfra`. Their source and the
scoring README are saved as text in `notes/` so the next notebook can call them directly for new shapes; the function
signatures are printed here.

In [ ]:
VDIR = COMP / 'verifiers' / 'dfthcsos'
for p in sorted(VDIR.rglob('*')):
    if p.suffix in ('.py', '.md') and p.is_file():
        (NOTES / ('src_' + p.relative_to(VDIR).as_posix().replace('/', '__') + '.txt')).write_text(p.read_text())
from dfthcsos.scoreinfra import costs_prepare
for mod in (costs_prepare, official_math, v_dfthcsos):
    print('\n###', mod.__name__)
    for name, fn in inspect.getmembers(mod, inspect.isfunction):
        if fn.__module__ == mod.__name__:
            try:
                print(f'  {name}{inspect.signature(fn)}')
            except (TypeError, ValueError):
                print('  ', name)
readme = VDIR / 'SCORING_README.md'
if readme.exists():
    print('\n' + readme.read_text()[:6000])

## The two-body target in pair space

As in our notebook 02 (whose parity with the official evaluator was checked to 1e-8), the verifier's `error_2` is the
Frobenius norm of `Σ a aᵀ − M`, where `M` is the shifted two-electron tensor on the 2,926 symmetric orbital pairs and
each copy `a_rc = Σ_b W_rbc · (u_rb u_rbᵀ)` is a combination of its group's 143 decoded projectors. The cell rebuilds
`M` and the public reconstruction and checks the error against the official native evaluator before anything else.

In [ ]:
with np.load(PUBLIC / 'submission.npz', allow_pickle=False) as z:
    BASE = {k: np.array(z[k]) for k in z.files}
print('NPZ fields:', {k: (v.shape, str(v.dtype)) for k, v in BASE.items()})
BROT = register_layout([km.Circuit.from_file(str(PUBLIC / 'angle_k_0.kmb'), format='kmb')]).brot
INPUTS = v_dfthcsos.load_inputs(PUBLIC / 'submission.npz')
REF, SHIFT = INPUTS.ref, INPUTS.S
N = int(REF['orbitals'])
VECTORS = official_math.reconstruct_u_vectors(BASE['angle_words'] / 2**(BROT - 1), BASE['tree_edges'])
VECTORS /= np.linalg.norm(VECTORS, axis=1, keepdims=True)
R, B, C = BASE['W'].shape
BODY = VECTORS[N:].reshape(R, B, N)

ii, jj = np.tril_indices(N)
d = np.sqrt(np.where(ii == jj, 1., 2.))
L = len(ii)
PROJ = BODY[:, :, ii] * BODY[:, :, jj] * d                     # (R, B, L): packed projectors per group
M = REF['tei'][ii[:, None], jj[:, None], ii[None, :], jj[None, :]].copy()
diag = (ii == jj).astype(float); shifted = SHIFT[ii, jj]
M += .5 * (shifted[:, None] * diag[None, :] + diag[:, None] * shifted[None, :])
M *= d[:, None] * d[None, :]
M = .5 * (M + M.T)
NORM_M2 = float(np.sum(M * M))

def error_of(factors):
    """||F Fᵀ − M||_F for a list of (L, k) factors, without forming F Fᵀ."""
    F = np.concatenate([f for f in factors if f.shape[1]], axis=1)
    cross = float(np.einsum('lk,lk->', F, M @ F))
    gram = F.T @ F
    return math.sqrt(max(NORM_M2 - 2 * cross + float(np.sum(gram * gram)), 0.))

A0 = [np.einsum('bc,bl->lc', BASE['W'][r], PROJ[r]) for r in range(R)]   # current copies, (L, C) per group
native = official_math.evaluate_dfthc_angle_words(
    BASE['angle_words'], BASE['W'], BASE['H2_I'] - BASE['W'].sum(axis=1), SHIFT, INPUTS.beta_1, REF,
    brot=BROT, tree_edges=BASE['tree_edges'], energy_upper=reference.UPPER_ENERGY)
e2_ours = error_of(A0)
print(f'shape (R, B, C) = {(R, B, C)}, pairs L = {L}')
print(f"error_2: native {1e3 * native['error_2']:.6f} mHa | pair-space rebuild {1e3 * e2_ours:.6f} mHa")
print(f"native lambda_eff {native['lambda_eff']:.6f}, Lambda {native['Lambda']:.4f}, E_gap {native['E_gap']:.6f}, error_1 {1e3 * native['error_1']:.4f} mHa")
assert abs(e2_ours - native['error_2']) < 1e-7, 'target rebuild disagrees with the official evaluator'
print('target rebuild matches the official evaluator')

## Spectral floor for any factorization

Every DFTHC reconstruction is `Σ a aᵀ`: positive semidefinite with rank at most `R·C`. The best such matrix of rank `k`
keeps the `k` largest positive eigenvalues of `M`, so no factorization with `R·C = k` copies, whatever its vectors,
can have `error_2` below `sqrt(Σ negative λ² + Σ discarded positive λ²)`. The floor ignores the 8-bit words and the
shared dictionaries, so the true minimum is higher. It is a necessary condition, not a promise.

In [ ]:
t0 = time.time()
EIG = np.linalg.eigvalsh(M)[::-1]
pos = EIG[EIG > 0]; neg = EIG[EIG < 0]
neg2 = float(np.sum(neg ** 2))
tail = np.r_[np.cumsum((pos ** 2)[::-1])[::-1], 0.]           # tail[k] = Σ_{i≥k} pos_i²
def floor(k): return math.sqrt(neg2 + tail[min(k, len(pos))])
print(f'eigendecomposition {time.time() - t0:.1f} s: {len(pos)} positive, {len(neg)} negative eigenvalues; '
      f'negative part alone {1e3 * math.sqrt(neg2):.3f} mHa')
rows = []
for k in [60, 90, 120, 150, 180, 200, 240, 270, 300, 330, 360, 400, 450, 500, 600]:
    f = floor(k)
    rows.append(dict(RC=k, floor_mHa=round(1e3 * f, 3),
                     fits_20=f < ERROR_LIMIT,
                     leaves_for_rounding_mHa=round(1e3 * math.sqrt(max(ERROR_LIMIT ** 2 - f ** 2, 0)), 2)))
SPECTRAL = pd.DataFrame(rows)
print(SPECTRAL.to_string(index=False))
k20 = next((k for k in range(1, len(pos) + 1) if floor(k) < ERROR_LIMIT), None)
k_round = next((k for k in range(1, len(pos) + 1) if math.hypot(floor(k), ROUNDING_NOISE) < ERROR_LIMIT), None)
print(f'\nsmallest R·C with floor < 20 mHa: {k20}')
print(f'smallest R·C whose floor still leaves room for 8-bit rounding noise ({1e3 * ROUNDING_NOISE:.2f} mHa): {k_round}')
print(f'current R·C = {R * C}: floor {1e3 * floor(R * C):.3f} mHa against the achieved {1e3 * native["error_2"]:.3f} mHa')

## Optimal error with the current angle words

The angle circuits fix the vectors `u_rb`, so each group's copies live in the span `S_r` of its 143 projectors; the
weights `W` and the copy count `C` are free and only state preparation depends on `C`. For a fixed dictionary the best
group update is exact: project the residual on `S_r` and keep its `C` largest positive eigenpairs. Block-coordinate
sweeps of that update give a monotone fit, an *achievable* `error_2` for each `C` with today's circuits, ignoring λ.
Then each group in turn is removed entirely, with the other eight refitted at `C ∈ {40, 45, 50}`: a shape `R = 8`
that would remove 143 vector addresses from every angle loader.

In [ ]:
QB = [np.linalg.qr(PROJ[r].T)[0] for r in range(R)]          # (L, B) orthonormal basis of each S_r
QMQ = [q.T @ (M @ q) for q in QB]

def truncate(f, k):
    """Best rank-k part of f fᵀ, as a factor."""
    if k >= f.shape[1]:
        return f.copy()
    u, s, _ = np.linalg.svd(f, full_matrices=False)
    return u[:, :k] * s[:k]

def bcd(start, ks, groups, sweeps=BCD_SWEEPS, label=''):
    F = {r: truncate(start[r], ks[r]) if start[r] is not None else np.zeros((L, 0)) for r in groups}
    history = [error_of(list(F.values()))]
    for sweep in range(sweeps):
        for r in groups:
            q = QB[r]
            K = QMQ[r].copy()
            for s in groups:
                if s != r and F[s].shape[1]:
                    t = q.T @ F[s]
                    K -= t @ t.T
            K = .5 * (K + K.T)
            val, vec = np.linalg.eigh(K)
            top = np.argsort(val)[::-1][:ks[r]]
            top = top[val[top] > 0]
            F[r] = (q @ vec[:, top]) * np.sqrt(val[top])
        history.append(error_of(list(F.values())))
        if history[-2] - history[-1] < 1e-9:
            break
    return F, history

t0 = time.time()
rows = []
F40 = None
for c in C_GRID:
    F, hist = bcd(A0, {r: c for r in range(R)}, list(range(R)))
    if c == C:
        F40 = F
    rows.append(dict(R=R, B=B, C=c, RC=R * c, start_mHa=round(1e3 * hist[0], 3), optimal_mHa=round(1e3 * hist[-1], 3),
                     sweeps=len(hist) - 1, fits_20=hist[-1] < ERROR_LIMIT, spectral_floor_mHa=round(1e3 * floor(R * c), 3)))
    print(rows[-1], f'{time.time() - t0:.0f} s', flush=True)
FIXED_WORDS = pd.DataFrame(rows)

In [ ]:
t0 = time.time()
rows = []
for drop in range(R):
    keep = [r for r in range(R) if r != drop]
    for c in DROP_C_GRID:
        F, hist = bcd({r: (A0[r] if r != drop else None) for r in range(R)},
                      {r: c for r in keep}, keep)
        rows.append(dict(dropped_group=drop, R=R - 1, C=c, RC=(R - 1) * c, optimal_mHa=round(1e3 * hist[-1], 3),
                         fits_20=hist[-1] < ERROR_LIMIT, spectral_floor_mHa=round(1e3 * floor((R - 1) * c), 3)))
    print(f'group {drop}:', [r['optimal_mHa'] for r in rows[-len(DROP_C_GRID):]], f'{time.time() - t0:.0f} s', flush=True)
DROP_GROUP = pd.DataFrame(rows)
print(DROP_GROUP.sort_values('optimal_mHa').head(12).to_string(index=False))

## Official resource probe

How much does the verifier actually charge less for fewer copies with the same circuits? For each `C` the weights are
truncated per group to their best rank-`C` part (an SVD of `[W; H2_I]`, keeping its Gram matrix as closely as
possible) and the folder goes through the unmodified `verify`. The accuracy status may be `FAIL`, since nothing is
refitted; the point is the exact T, Q and resource breakdown for each shape.

In [ ]:
def truncated_npz(c: int) -> dict:
    data = dict(BASE)
    a = np.concatenate([BASE['W'], BASE['H2_I'][:, None, :]], axis=1)       # (R, B+1, C)
    out = []
    for g in a:
        u, s, _ = np.linalg.svd(g, full_matrices=False)
        out.append(u[:, :c] * s[:c])
    out = np.stack(out)
    data['W'], data['H2_I'] = out[:, :-1], out[:, -1]
    if 'offset_W' in data:
        data['offset_W'] = data['H2_I'] - data['W'].sum(axis=1)
    return data

rows = []
for c in PROBE_C:
    folder = ROOT / f'probe_C{c}'
    if folder.exists():
        shutil.rmtree(folder)
    shutil.copytree(PUBLIC, folder)
    if c != C:
        np.savez(folder / 'submission.npz', **truncated_npz(c))
    t0 = time.time()
    try:
        r = v_dfthcsos.verify(str(folder), salt=SALT)
    except Exception as exc:
        r = dict(status='ERROR', reason=repr(exc)[:300])
    rec = dict(C=c, seconds=round(time.time() - t0), **short(r))
    res = r.get('resources') or {}
    for k, v in res.items():
        if isinstance(v, (int, float)):
            rec['res_' + k] = v
    if r.get('status') != 'PASS':
        rec['reason'] = str(r.get('reason') or r.get('failures') or '')[:200]
    rows.append(rec)
    print({k: rec[k] for k in rec if not k.startswith('res_')}, flush=True)
    shutil.rmtree(folder)
PROBE = pd.DataFrame(rows)
print(PROBE.to_string(index=False))

## What a feasible shape would be worth

Leaderboard value of a chemistry cost `X`: the task ratio is `REFERENCE / X` and the portfolio score the geometric mean
with our other two ratios. For `C`-only changes the exact T and Q come from the probe above, with λ held at today's
value (an optimistic assumption). For group removal the T saving is a rough scaling of Dmitrii Gluzdov's breakdown:
angle loaders 9,492 Toffolis over 1,363 addresses, preparation 4,171 proportional to `R·C`, the rest fixed.

In [ ]:
LAM, T0, Q0 = float(BASE_RESULT['lambda_eff']), float(BASE_RESULT['toffoli_count']), float(BASE_RESULT['num_qubits'])
def lb(cost):
    ratio = REFERENCE_COST / cost
    return ratio, (ratio * PORTFOLIO['invsqrt'] * PORTFOLIO['dqi']) ** (1 / 3)
base_ratio, base_lb = lb(LAM * T0 * Q0)
print(f'baseline: λ {LAM:.4f} T {T0:,.0f} Q {Q0:.0f} cost {LAM * T0 * Q0:,.0f} ratio {base_ratio:.4f} LB {base_lb:.5f}')

rows = []
for rec in PROBE.to_dict('records'):
    t, q = rec.get('toffoli_count'), rec.get('num_qubits')
    if t is None or q is None or (isinstance(t, float) and math.isnan(t)):
        continue
    fw = FIXED_WORDS.set_index('C').get('optimal_mHa', pd.Series()).get(rec['C'])
    ratio, score = lb(LAM * t * q)
    rows.append(dict(shape=f"(9,143,{rec['C']})", source='official probe', T=t, Q=q, optimal_error_2_mHa=fw,
                     feasible=bool(fw is not None and fw < 1e3 * ERROR_LIMIT), task_ratio=round(ratio, 4), LB=round(score, 5)))

ADDR = 76 + R * B
for drop_c in DROP_C_GRID:
    best = DROP_GROUP[DROP_GROUP.C == drop_c].sort_values('optimal_mHa').iloc[0]
    t = 9492 * (76 + (R - 1) * B) / ADDR + 4171 * ((R - 1) * drop_c) / (R * C) + (T0 - 9492 - 4171)
    ratio, score = lb(LAM * t * Q0)
    rows.append(dict(shape=f'(8,143,{drop_c}) drop g{int(best.dropped_group)}', source='rough model', T=round(t), Q=Q0,
                     optimal_error_2_mHa=best.optimal_mHa, feasible=bool(best.fits_20),
                     task_ratio=round(ratio, 4), LB=round(score, 5)))
VALUE = pd.DataFrame(rows)
print(VALUE.to_string(index=False))

## Verdict

The cell below states the decision rule from the plan and applies it. A shape is worth a training notebook if its
achievable `error_2` leaves room under 20 mHa (fixed-word fits do not need new rounding; new words would need about
14 mHa of it) **and** it lifts the chemistry ratio by at least 15%.

In [ ]:
print('Spectral floor: smallest R·C under 20 mHa =', k20, '| with room for 8-bit rounding =', k_round)
print(f'Current shape R·C = {R * C}: floor {1e3 * floor(R * C):.2f} mHa, fixed-word optimum {FIXED_WORDS.iloc[0].optimal_mHa} mHa, achieved {1e3 * native["error_2"]:.3f} mHa')
fw_ok = FIXED_WORDS[FIXED_WORDS.fits_20]
print('Fixed words: smallest C that fits 20 mHa =', int(fw_ok.C.min()) if len(fw_ok) else None)
dg_ok = DROP_GROUP[DROP_GROUP.fits_20]
print('Dropping one group: feasible variants =', len(dg_ok), '' if dg_ok.empty else dg_ok.sort_values('optimal_mHa').head(3).to_dict('records'))
worth = VALUE[(VALUE.feasible) & (VALUE.task_ratio >= 1.15 * base_ratio)]
print('\nShapes worth training (feasible and >= 1.15x chemistry ratio):')
print(worth.to_string(index=False) if len(worth) else '  none: chemistry shape changes within reach of the current words do not pay ×1.15')

SUMMARY = dict(baseline=short(BASE_RESULT), spectral_k20=k20, spectral_k_rounding=k_round,
               spectral=SPECTRAL.to_dict('records'), fixed_words=FIXED_WORDS.to_dict('records'),
               drop_group=DROP_GROUP.to_dict('records'), probe=PROBE.to_dict('records'), value=VALUE.to_dict('records'),
               minutes=round((time.time() - NB_T0) / 60, 1))
(ROOT / 'shape_screen.json').write_text(json.dumps(SUMMARY, indent=1, default=float))
print(f"\nSHAPE SCREEN DONE | {SUMMARY['minutes']} minutes | results in /kaggle/working/shape_screen.json and notes/")

# Sources and credits

- **Lera (marculera)**: the `(9, 143, 40)` compact-group factorization, its angle words and loaders, and the rounding
  numbers used here (continuous 13.39 → rounded 19.46 mHa at C = 40):
  [DF-THC 10x: Compact Groups From the Hamiltonian](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian).
  Lera credits Michael Rustad's 8-bit workspace cliff at B ≥ 129, SomeJoint's coded bits, mrmeows' emitter,
  lightningv08's five-word batches and Ahmed Berat Özer's penalty and rounding work.
- **Dmitrii Gluzdov**: the `k = 0, 14` loaders and the Toffoli breakdown used in the rough model (ANGLE 9,492,
  inner preparation 4,171, outer 320, Givens 2,250):
  [Quantum Circuit Compilation from Source](https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source).
- **heliosli**: the refitted weights and the public release dataset
  [qcc-circuits-live](https://www.kaggle.com/datasets/heliosli/qcc-circuits-live); **yoyobar** for the mirror release.
- **Ahmed Berat Özer**: the 7-bit estimate and the evidence that fixed-word refits are saturated:
  [Quantum AI – DF-THC Group Compression](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression).
- **Lynn Langit**: the λ_eff lower bound from the gap floor
  ([discussion 753633](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/discussion/753633)).
- **theredbluepill**: the suggestion to try narrower chemistry words
  ([discussion 753738](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/discussion/753738)).
- **Guang Hao Low et al.**, DFTHC with spectral amplification, [arXiv:2502.15882](https://arxiv.org/abs/2502.15882);
  **Tanuj Khattar, Nicholas C. Rubin, Craig Gidney, Michael Broughton, Guang Hao Low and Ryan Babbush**, the competition
  specification and verifiers, [arXiv:2610.23832](https://arxiv.org/abs/2610.23832).
- Ours: the spectral floor, the fixed-dictionary block-coordinate fit, the group-removal screen and the probe. They
  reuse the pair-space target from our notebook 02, checked against the official evaluator.